### dash-cytoscape


In [1]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT0_DATA = ROOT0 / "data"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import pdwritecsv, pdreadcsv, create_dir, write_txt
from libs.enricher_lib import enricheR
from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [2]:
root0 = Path('/home/flavio/uv/perturb_agent')
root0_data = root0 / 'data'
root_colab = root0_data / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-ACC'

disease = PSI_ID

root_project = create_dir(root0_data, PROG_ID)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=root0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={ptw_min_num_of_degs_cut}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-ACC/config/all_lfc_cutoffs_TCGA-ACC.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [3]:
enr = enricheR(disease=disease, gene_protein=gene_protein, s_omics=s_omics, 
            project=project, s_project=s_project, 
            root0=root0, root0_data=root0_data,
            prog_id=PROG_ID, psi_id=PSI_ID,
            case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
            std_filename=std_filename, std_filename_list=std_filename_list,
            geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
            tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
            LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
            num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
            min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
            saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", enr.root0, enr.root_disease)
case = case_list[0]
print(">>>", case)

enr.cfg.set_default_best_lfc_cutoff(enr.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = enr.open_case_params(case, LFC_cut=1.0, lfc_FDR_cut=0.05,
						 ptw_pval_cut=0.05, ptw_FDR_cut=0.05, ptw_min_num_of_degs_cut=3, verbose=False)
print("\nEcho Parameters:")
print(enr.echo_parameters())


Start opening tables ....
Building synonym dictionary ...

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-ACC
>>> Tumor

Echo Parameters:
	12395/12395 DEGs/ensembl.
		Up 7131/7131 DEGs/ensembl.
		Dw 5264/5264 DEGs/ensembl.

Found 0 (best=0) pathways for geneset num=0 'Undefined'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=3No enrichment analysis was calculated.


In [4]:
enr.enr_db_list

['Reactome_Pathways_2024']

In [5]:
enr.dflfc_ori.head(2)

,ensembl_id,symbol,biotype,lfc,lfcSE,statistic,pval,fdr,baseMean,method,abs_lfc
0,ENSG00000234795,RFTN1P1,transcribed_unprocessed_pseudogene,26.457,2.951,8.966,3.086e-19,4.845e-18,121.435,DESeq2,26.457
1,ENSG00000138653,NDST4,protein_coding,23.413,2.952,7.930,2.189e-15,2.723e-14,13.755,DESeq2,23.413


In [6]:
_, _, dflfc = enr.list_of_degs(save_file=False, force=False, prompt_verbose=False, verbose=False)
print(len(dflfc))
dflfc.head(3)

12395


,ensembl_id,symbol,biotype,lfc,lfcSE,statistic,pval,fdr,baseMean,method,abs_lfc
0,ENSG00000265681,RPL17,protein_coding,-8.681,0.210,-41.345,0.0,0.0,18399.050,DESeq2,8.681
1,ENSG00000170296,GABARAP,protein_coding,-6.174,0.149,-41.461,0.0,0.0,7919.843,DESeq2,6.174
2,ENSG00000178127,NDUFV2,protein_coding,-8.456,0.192,-44.112,0.0,0.0,2354.304,DESeq2,8.456


In [7]:
def calc_enrichment_analysis(verbose: bool = False, force: bool = False):

    _, _, dflfc = enr.list_of_degs(save_file=False, force=False, prompt_verbose=False, verbose=verbose)

    print(">>> dflfc", len(dflfc))

    df2 = dflfc[dflfc.biotype.isin(enr.biotype_annot)]
    df2 = df2.sort_values('fdr', ascending=True)
    
    degs_to_reactome = df2.symbol.to_list()[:enr.MAX_GENES_ENRICHR_SAFE]

    print(">>> degs_to_reactome", len(degs_to_reactome))
    
    enr.calc_EA_dataset_symbol(list(degs_to_reactome), calc_many_sig=False, default=False, force=force, verbose=verbose)
    return enr.df_enr0

In [8]:
enr.set_db(0, verbose=True)

>>> Reactome_Pathways_2024


In [11]:
df_enr0 = calc_enrichment_analysis(verbose=True, force=False)
df_enr0.head()

>>> dflfc 12395
>>> degs_to_reactome 2000


,pathway,pathway_id,pval,fdr,odds_ratio,combined_score,genes,num_of_genes
0,Condensation of Prometaphase Chromosomes,R-HSA-2514853,1.234e-06,0.002,24.092,327.784,"['CCNB2', 'CCNB1', 'CSNK2A2', 'CSNK2B', 'CDK1', 'NCAPG', 'NCAPD2', 'SMC4']",8
1,Cell Cycle Checkpoints,R-HSA-69620,4.959e-05,0.028,2.000,19.825,"['BLM', 'ANAPC15', 'CDCA8', 'MCM10', 'PKMYT1', 'PMF1', 'SKA1', 'CCNB2', 'CCN...",47
2,Resolution of Sister Chromatid Cohesion,R-HSA-2500257,1.055e-04,0.028,2.474,22.659,"['CDCA5', 'CDCA8', 'PMF1', 'SKA1', 'CCNB2', 'CCNB1', 'TUBA1B', 'NUF2', 'BUB1...",27
3,Mitotic Spindle Checkpoint,R-HSA-69618,8.053e-05,0.028,2.637,24.856,"['ANAPC15', 'CDCA8', 'PMF1', 'SKA1', 'NUF2', 'BUB1', 'CENPU', 'UBE2C', 'UBE2...",25
4,Amplification of Signal From Unattached Kinetochores via a MAD2 Inhibitory S...,R-HSA-141444,1.162e-04,0.028,2.769,25.093,"['CENPU', 'CDCA8', 'PPP2R5D', 'KNL1', 'PMF1', 'SKA1', 'NDC80', 'SGO1', 'CENP...",22


In [14]:
row = enr.df_enr0.iloc[0]

pathway = row.pathway
pathway_id = row.pathway_id

pathway_id, pathway

('R-HSA-2514853', 'Condensation of Prometaphase Chromosomes')

### Expression table

In [15]:
from libs.tcga_gdc_lib import GDC

gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA)

In [16]:
df_psi = gdc.get_primary_sites(PROG_ID)

In [17]:
gdc.set_primary_site(PSI_ID)

True

In [18]:
verbose=False
df_tumor, df_normal, df_gtex_ctrl = gdc.get_file_expression_both_tumor_and_normal(verbose=verbose)

Dowloading normal files: 
Dowloading tumor files: 0..........10..........20..........30..........40..........50..........60..........70..........80...
Preparing GTEx metadata...
GTEx metadata prepared on df_meta_prep length: 138


#### Open DASH_CYTO passin dflfc_ori

In [22]:
ROOT_COLAB = ROOT0_DATA / 'colab'
ROOT_OWL = ROOT_COLAB / 'owl'
ROOT_REACTOME = ROOT_COLAB / 'reactome'

from libs.reactome_lib import Reactome

rea = Reactome(root_owl=ROOT_OWL, root_reactome=ROOT_REACTOME)

df_gmt = rea.open_reactome_gmt()

In [23]:
dfa = rea.df_gmt.loc[rea.df_gmt['pathway_id'] == pathway_id]
pathway_genes = [] if dfa.empty else dfa.iloc[0]['genes']
if isinstance(pathway_genes, str):
    pathway_genes = list(eval(pathway_genes))
    pathway_genes.sort()

In [24]:
dcy = DASH_CYTO(root0=ROOT0, root0_data=root0_data, dflfc_ori=enr.dflfc_ori, 
                lfc_cutoff=1.0, fdr_cutoff=0.05, 
                found_degs=enr.degs, pathway_genes=pathway_genes)

In [26]:
'''
owl_file = "R-HSA-165159_level3.owl"
owl_file = "R-HSA-114608_level3.owl"

pathway = 'Platelet degranulation'
pathway_id = "R-HSA-114608"

pathway = 'Cell Cycle Checkpoints'
pathway_id = "R-HSA-69620"

pathway = 'Condensation of Prometaphase Chromosomes'
pathway_id = "R-HSA-2514853"
'''

ret = dcy.read_owl(pathway_id, pathway)
print(ret)
rdf = dcy.rdf


True


### Dash

In [ ]:
height = "95%"
width = "100%"
marginTop="20px"

dcy.run_app(height=height, width=width, marginTop=marginTop, port=8051)